In [3]:
import requests
import zipfile
import pandas as pd
from pathlib import Path

In [6]:
PASTA_RAW = Path("../data/raw")
PASTA_RAW.mkdir(parents=True, exist_ok=True)
print(f"Pasta de dados: {PASTA_RAW.resolve()}")

Pasta de dados: /Users/brendadscc/brasil_publico/data/raw


In [8]:
def baixar_arquivo(url, destino):
    destino = Path(destino)

    if destino.exists():
        print(f"Arquivo já existe, pulando download: {destino.name}")
        return destino
    print(f"Baixando {destino.name}...")
    print("(isso pode levar alguns minutos - arquivo grande)")
    
    resposta = requests.get(url, stream=True, timeout=120)

    resposta.raise_for_status()

    with open(destino, "wb") as arquivo:
        for pedaco in resposta.iter_content(chunk_size=1024 * 1024):
            arquivo.write(pedaco)

    tamanho_mb = destino.stat().st_size / 1_000_000
    print(f"Download concluído: {tamanho_mb:.1f} MB")

    return destino

In [9]:
URL_2024 = "https://download.inep.gov.br/microdados/microdados_censo_da_educacao_superior_2024.zip"

arquivo_zip = baixar_arquivo(URL_2024, PASTA_RAW / "censo_2024.zip")


Baixando censo_2024.zip...
(isso pode levar alguns minutos - arquivo grande)
Download concluído: 43.4 MB


In [12]:
def extrair_zip(caminho_zip, pasta_destino):
    caminho_zip = Path(caminho_zip)
    pasta_destino = Path(pasta_destino)
    pasta_destino.mkdir(exist_ok=True)
    
    with zipfile.ZipFile(caminho_zip) as zf:
        print("O que tem dentro do ZIP:")
        for nome in zf.namelist():
            print(f"  {nome}")
            
        zf.extractall(pasta_destino)

    print(f"\nExtraído em: {pasta_destino}")
    return pasta_destino

pasta_censo = extrair_zip(
    PASTA_RAW / "censo_2024.zip",
    PASTA_RAW / "censo_2024"
)

O que tem dentro do ZIP:
  microdados_censo_da_educacao_superior_2024/
  microdados_censo_da_educacao_superior_2024/Anexos/
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO I - Dicionário de Dados/
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO I - Dicionário de Dados/dicionário_dados_educação_superior.xlsx
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO I - Dicionário de Dados/Thumbs.db
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO I - Dicionário de Dados/~$Dicionário de Dados da Educação Básica.xlsx
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO I - Dicionário de Dados/~$dicionário_dados_educação_superior.xlsx
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO II - Questionários do Censo da Educação Superior/
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO II - Questionários do Censo da Educação Superior/ANEXO II - MÓDULO ALUNO 2024.pdf
  microdados_censo_da_educacao_superior_2024/Anexos/ANEXO II - Questionários do

In [16]:
csvs = list(pasta_censo.rglob("*.CSV")) + list(pasta_censo.rglob("*.csv"))

print(f"Total de CSVs encontrados: {len(csvs)}\n")
for i, csv in enumerate(csvs):
    tamanho_mb = csv.stat().st_size / 1_000_000
    print(f"[{i}]  {csv.name}  —  {tamanho_mb:.0f} MB")

Total de CSVs encontrados: 2

[0]  MICRODADOS_ED_SUP_IES_2024.CSV  —  1 MB
[1]  MICRODADOS_CADASTRO_CURSOS_2024.CSV  —  453 MB


In [17]:
caminho_csv = csvs[1]

cabecalho = pd.read_csv(
    caminho_csv,
    nrows=0,
    sep=";",
    encoding="latin-1"
)

print(f"Total de colunas no arquivo: {len(cabecalho.columns)}")
print("\nTodas as colunas disponíveis:")
for col in cabecalho.columns:
    print(f"  {col}")

Total de colunas no arquivo: 223

Todas as colunas disponíveis:
  NU_ANO_CENSO
  NO_REGIAO
  CO_REGIAO
  NO_UF
  SG_UF
  CO_UF
  NO_MUNICIPIO
  CO_MUNICIPIO
  IN_CAPITAL
  TP_DIMENSAO
  TP_ORGANIZACAO_ACADEMICA
  TP_REDE
  TP_CATEGORIA_ADMINISTRATIVA
  IN_COMUNITARIA
  IN_CONFESSIONAL
  CO_IES
  NO_CURSO
  CO_CURSO
  NO_CINE_ROTULO
  CO_CINE_ROTULO
  CO_CINE_AREA_GERAL
  NO_CINE_AREA_GERAL
  CO_CINE_AREA_ESPECIFICA
  NO_CINE_AREA_ESPECIFICA
  CO_CINE_AREA_DETALHADA
  NO_CINE_AREA_DETALHADA
  TP_GRAU_ACADEMICO
  IN_GRATUITO
  TP_MODALIDADE_ENSINO
  TP_NIVEL_ACADEMICO
  QT_CURSO
  QT_VG_TOTAL
  QT_VG_TOTAL_DIURNO
  QT_VG_TOTAL_NOTURNO
  QT_VG_TOTAL_EAD
  QT_VG_NOVA
  QT_VG_PROC_SELETIVO
  QT_VG_REMANESC
  QT_VG_PROG_ESPECIAL
  QT_INSCRITO_TOTAL
  QT_INSCRITO_TOTAL_DIURNO
  QT_INSCRITO_TOTAL_NOTURNO
  QT_INSCRITO_TOTAL_EAD
  QT_INSC_VG_NOVA
  QT_INSC_PROC_SELETIVO
  QT_INSC_VG_REMANESC
  QT_INSC_VG_PROG_ESPECIAL
  QT_ING
  QT_ING_FEM
  QT_ING_MASC
  QT_ING_DIURNO
  QT_ING_NOTURNO
  QT_ING

In [18]:
COLUNAS = [
    "NU_ANO_CENSO",
    "TP_MODALIDADE_ENSINO",
    "CO_REGIAO",
    "TP_GRAU_ACADEMICO",
    "TP_REDE",
    "QT_MAT",
    "QT_MAT_18_24",
    "QT_MAT_25_29",
    "QT_MAT_30_34",
    "QT_CONC",
]

df = pd.read_csv(
    caminho_csv,
    sep=";",
    encoding="latin-1",
    usecols=COLUNAS,
    low_memory=False
)

print(f"Linhas carregadas: {len(df):,}")
print(f"Memória usada: {df.memory_usage(deep=True).sum() / 1_000_000:.0f} MB")
df.head()

Linhas carregadas: 720,349
Memória usada: 58 MB


,NU_ANO_CENSO,CO_REGIAO,TP_REDE,TP_GRAU_ACADEMICO,TP_MODALIDADE_ENSINO,QT_MAT,QT_MAT_18_24,QT_MAT_25_29,QT_MAT_30_34,QT_CONC
0,2024,NaN,2,NaN,2,17,0,1,7,0
1,2024,NaN,2,NaN,2,0,0,0,0,0
2,2024,NaN,2,NaN,2,0,0,0,0,0
3,2024,NaN,2,NaN,2,0,0,0,0,0
4,2024,NaN,2,NaN,2,24,8,5,4,0


In [20]:
resultado = (
    df.groupby("TP_MODALIDADE_ENSINO")["QT_MAT"]
    .sum()
    .reset_index()
)

resultado["modalidade"] = resultado["TP_MODALIDADE_ENSINO"].map({
    1: "Presencial",
    2: "EAD"
})

total_geral = resultado["QT_MAT"].sum()
resultado["percentual"] = (resultado["QT_MAT"] / total_geral * 100).round(1)

print("=== Matrículas no ensino superior — Brasil 2024 ===\n")
for _, linha in resultado.iterrows():
    print(f"{linha['modalidade']:12}  {linha['QT_MAT']:>12,.0f} matrículas  ({linha['percentual']}%)")
print(f"\n{'TOTAL':12}  {total_geral:>12,.0f}")

=== Matrículas no ensino superior — Brasil 2024 ===

Presencial       5,037,875 matrículas  (49.3%)
EAD              5,189,391 matrículas  (50.7%)

TOTAL           10,227,266


In [21]:
faixa_etaria = (
    df.groupby("TP_MODALIDADE_ENSINO")[
        ["QT_MAT_18_24", "QT_MAT_25_29", "QT_MAT_30_34"]
    ]
    .sum()
)

faixa_etaria.index=faixa_etaria.index.map({1: "Presencial", 2: "EAD"})

faixa_pct = faixa_etaria.div(faixa_etaria.sum(axis=1), axis=0) * 100
faixa_pct = faixa_pct.round(1)

print("=== Distribuição etária por modalidade (% dentro de cada uma) === \n")
print(faixa_pct.to_string())
print("\nInterpretação: se EAD tem proporção maior de 25+ anos,")
print("sugere que atinge adultos que trabalham - não migração do presencial.")

=== Distribuição etária por modalidade (% dentro de cada uma) === 

                      QT_MAT_18_24  QT_MAT_25_29  QT_MAT_30_34
TP_MODALIDADE_ENSINO                                          
Presencial                    71.5          20.5           8.0
EAD                           41.3          32.2          26.4

Interpretação: se EAD tem proporção maior de 25+ anos,
sugere que atinge adultos que trabalham - não migração do presencial.


In [23]:
conc = (
    df.groupby("TP_MODALIDADE_ENSINO")[["QT_MAT", "QT_CONC"]]
    .sum()
    .reset_index()
)

conc["modalidade"] = conc["TP_MODALIDADE_ENSINO"].map({
    1: "Presencial",
    2: "EAD"
})

conc["taxa_conclusao"] = (conc["QT_CONC"] / conc["QT_MAT"] * 100).round(1)

print("=== Matriculados vs Concluintes por modalidade - 2024 ===\n")

for _, linha in conc.iterrows():
    print(f"{linha['modalidade']:12}  "
          f"Matrículas: {linha['QT_MAT']:>10,.0f}  "
          f"Concluintes: {linha['QT_CONC']:>9,.0f}   "
          f"Taxa: {linha['taxa_conclusao']}%")

print("\nNota: taxa de conclusão aqui é um proxy, não evasão real.")
print("Para evasão real precisaríamos acompanhar a mesma turma ao longo dos anos.")
          


=== Matriculados vs Concluintes por modalidade - 2024 ===

Presencial    Matrículas:  5,037,875  Concluintes:   729,246   Taxa: 14.5%
EAD           Matrículas:  5,189,391  Concluintes:   604,742   Taxa: 11.7%

Nota: taxa de conclusão aqui é um proxy, não evasão real.
Para evasão real precisaríamos acompanhar a mesma turma ao longo dos anos.


In [ ]:
grau = (
    df.groupby(["TP_MODALIDADE_ENSINO", "TP_GRAU_ACADEMICO"])[

In [28]:
grau = (
    df.groupby(["TP_MODALIDADE_ENSINO", "TP_GRAU_ACADEMICO"])[["QT_MAT", "QT_CONC"]]
    .sum()
    .reset_index()
)

grau["modalidade"] = grau["TP_MODALIDADE_ENSINO"].map({
    1: "Presencial",
    2: "EAD"
})

grau["grau"] = grau["TP_GRAU_ACADEMICO"].map({
    1: "Bacharelado",
    2: "Licenciatura",
    3: "Tecnológico"
})

grau["taxa_conclusao"] = (grau["QT_CONC"] / grau["QT_MAT"] * 100).round(1)
grau["share_ead"] = None  

total_por_grau = df.groupby("TP_GRAU_ACADEMICO")["QT_MAT"].sum()

print("=== Matrículas e conclusão por grau e modalidade ===\n")
for g_nome in ["Bacharelado", "Licenciatura", "Tecnológico"]:
    print(f"── {g_nome} ──")
    subset = grau[grau["grau"] == g_nome][["modalidade", "QT_MAT", "QT_CONC", "taxa_conclusao"]]
    
    total_g = subset["QT_MAT"].sum()
    for _, row in subset.iterrows():
        share = row["QT_MAT"] / total_g * 100
        print(f"  {row['modalidade']:12}  "
              f"Matrículas: {row['QT_MAT']:>9,.0f}  ({share:.0f}%)  "
              f"Conclusão: {row['taxa_conclusao']}%")
    print()
        

=== Matrículas e conclusão por grau e modalidade ===

── Bacharelado ──
  Presencial    Matrículas: 4,101,368  (64%)  Conclusão: 14.2%
  EAD           Matrículas: 2,291,425  (36%)  Conclusão: 8.3%

── Licenciatura ──
  Presencial    Matrículas:   541,262  (31%)  Conclusão: 13.3%
  EAD           Matrículas: 1,177,328  (69%)  Conclusão: 11.4%

── Tecnológico ──
  Presencial    Matrículas:   356,970  (17%)  Conclusão: 21.4%
  EAD           Matrículas: 1,697,467  (83%)  Conclusão: 16.5%



In [33]:
faixa = (
    df.groupby("TP_MODALIDADE_ENSINO")[
        ["QT_MAT_18_24", "QT_MAT_25_29", "QT_MAT_30_34"]
    ]
    .sum()
)

faixa.index = faixa.index.map({1: "Presencial", 2: "EAD"})

faixa_pct = faixa.div(faixa.sum(axis=1), axis=0) * 100

print("=== Distribuição etária — % dentro de cada modalidade ===\n")
print(f"{'Faixa':>12}    {'Presencial':>12} {'EAD':>12}")
print("-" * 45)
for col, nome in [("QT_MAT_18_24","18–24 anos"), ("QT_MAT_25_29","25–29 anos"), ("QT_MAT_30_34","30–34 anos")]:
    pres = faixa_pct.loc["Presencial", col]
    ead  = faixa_pct.loc["EAD", col]
    print(f"{nome:>12}  {pres:>11.1f}%  {ead:>11.1f}%")

print("\nNota: colunas não somam 100% — há faixas acima de 34 anos não incluídas aqui.")
                  

=== Distribuição etária — % dentro de cada modalidade ===

       Faixa      Presencial          EAD
---------------------------------------------
  18–24 anos         71.5%         41.3%
  25–29 anos         20.5%         32.2%
  30–34 anos          8.0%         26.4%

Nota: colunas não somam 100% — há faixas acima de 34 anos não incluídas aqui.


In [37]:
rede = (
    df.groupby(["TP_MODALIDADE_ENSINO", "TP_REDE"])["QT_MAT"]
    .sum()
    .reset_index()
)

rede["modalidade"] = rede["TP_MODALIDADE_ENSINO"].map({1: "Presencial", 2: "EAD"})
rede["rede"]       = rede["TP_REDE"].map({1: "Pública", 2: "Privada"})

print("=== Distribuição pública vs privada por modalidade ===\n")
for modalidade in ["Presencial", "EAD"]:
    subset = rede[rede["modalidade"] == modalidade]
    total  = subset["QT_MAT"].sum()
    print(f"── {modalidade} ──")
    for _, row in subset.iterrows():
        share = row["QT_MAT"] / total * 100
        print(f"  {row['rede']:10}  {row['QT_MAT']:>10,.0f}  ({share:.1f}%)")
    print()
                                        

=== Distribuição pública vs privada por modalidade ===

── Presencial ──
  Pública      1,849,728  (36.7%)
  Privada      3,188,147  (63.3%)

── EAD ──
  Pública        215,339  (4.1%)
  Privada      4,974,052  (95.9%)



In [41]:
import pandas as pd
from pathlib import Path

PASTA_CLEAN = Path("../data/clean")
PASTA_CLEAN.mkdir(exist_ok=True)

resumo_modalidade = (
    df.groupby("TP_MODALIDADE_ENSINO")[["QT_MAT", "QT_CONC"]]
    .sum()
    .reset_index()
)
resumo_modalidade["modalidade"] = resumo_modalidade["TP_MODALIDADE_ENSINO"].map({
    1: "Presencial", 2: "EAD"
})
resumo_modalidade["taxa_conclusao_pct"] = (
    resumo_modalidade["QT_CONC"] / resumo_modalidade["QT_MAT"] * 100
).round(1)
resumo_modalidade["share_total_pct"] = (
    resumo_modalidade["QT_MAT"] / resumo_modalidade["QT_MAT"].sum() * 100
).round(1)

faixa = (
    df.groupby("TP_MODALIDADE_ENSINO")[
        ["QT_MAT_18_24", "QT_MAT_25_29", "QT_MAT_30_34"]
    ]
    .sum()
    .reset_index()
)
faixa["modalidade"] = faixa["TP_MODALIDADE_ENSINO"].map({
    1: "Presencial", 2: "EAD"
})
total_por_modalidade = df.groupby("TP_MODALIDADE_ENSINO")["QT_MAT"].sum()
for col in ["QT_MAT_18_24", "QT_MAT_25_29", "QT_MAT_30_34"]:
    faixa[col + "_pct"] = (
        faixa[col] / faixa["TP_MODALIDADE_ENSINO"]
        .map(total_por_modalidade) * 100
    ).round(1)

grau = (
    df.groupby(["TP_MODALIDADE_ENSINO", "TP_GRAU_ACADEMICO"])[["QT_MAT", "QT_CONC"]]
    .sum()
    .reset_index()
)
grau["modalidade"] = grau["TP_MODALIDADE_ENSINO"].map({1: "Presencial", 2: "EAD"})
grau["grau"]       = grau["TP_GRAU_ACADEMICO"].map({
    1: "Bacharelado", 2: "Licenciatura", 3: "Tecnológico"
})
grau["taxa_conclusao_pct"] = (grau["QT_CONC"] / grau["QT_MAT"] * 100).round(1)

resumo_modalidade.to_csv(PASTA_CLEAN / "resumo_modalidade_2024.csv", index=False)
faixa.to_csv(PASTA_CLEAN / "resumo_faixa_etaria_2024.csv", index=False)
grau.to_csv(PASTA_CLEAN / "resumo_grau_academico_2024.csv", index=False)

print("Arquivos salvos em data/clean/:")
print("  resumo_modalidade_2024.csv")
print("  resumo_faixa_etaria_2024.csv")
print("  resumo_grau_academico_2024.csv")
print("\nFonte: INEP, Censo da Educação Superior 2024")
print("Acesso: https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-da-educacao-superior")

Arquivos salvos em data/clean/:
  resumo_modalidade_2024.csv
  resumo_faixa_etaria_2024.csv
  resumo_grau_academico_2024.csv

Fonte: INEP, Censo da Educação Superior 2024
Acesso: https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-da-educacao-superior
